# 05 — Кастомная сложность из OSM

Вычисляем сложность сегмента сами — без SAC-меток hikr.org.

Pipeline: скачать OSM (тропы + дороги + вода) для bbox hikr-данных → пространственный поиск по каждому midpoint → on_trail / on_road / near_water → custom_difficulty → обучить модель → сравнить с SAC-baseline.

In [ ]:
!pip install shapely

In [ ]:
import json, pickle, time, warnings
from pathlib import Path
from math import radians, cos, sqrt, degrees, atan2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

CACHE_DIR  = Path("cache")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
print("OK")


## 1. Данные hikr + bbox

In [ ]:
df_raw = pickle.load(open(CACHE_DIR / "hikr_model_df.pkl", "rb"))
n_all = df_raw['track_id'].nunique()
print(f"Всего сегментов: {len(df_raw):,}  треков: {n_all:,}")

# Оставляем только Альпы (92.6% данных, 15 тайлов вместо 4400)
ALPS_FILTER = dict(lat_min=43, lat_max=49, lon_min=5, lon_max=20)
alps_mask = (
    (df_raw["lat_mid"] >= ALPS_FILTER["lat_min"]) &
    (df_raw["lat_mid"] <= ALPS_FILTER["lat_max"]) &
    (df_raw["lon_mid"] >= ALPS_FILTER["lon_min"]) &
    (df_raw["lon_mid"] <= ALPS_FILTER["lon_max"])
)
df = df_raw[alps_mask].copy()
n_alp = df['track_id'].nunique()
print(f"Альпы: {len(df):,} сегментов  {n_alp:,} треков  ({100*alps_mask.mean():.1f}% данных)")

lat_min, lat_max = df["lat_mid"].min(), df["lat_mid"].max()
lon_min, lon_max = df["lon_mid"].min(), df["lon_mid"].max()
print(f"bbox: N={lat_max:.2f} S={lat_min:.2f} W={lon_min:.2f} E={lon_max:.2f}")
from math import radians, cos
area = (lat_max-lat_min)*111 * (lon_max-lon_min)*111*cos(radians((lat_max+lat_min)/2))
print(f"Площадь ~{area:.0f} км^2")


## 2. Скачать OSM (тайлами 3°×3°)

In [ ]:
OSM_CACHE = CACHE_DIR / "alps_osm.pkl"
PBF_PATH  = CACHE_DIR / "alps-latest.osm.pbf"
PBF_URL   = "https://download.geofabrik.de/europe/alps-latest.osm.pbf"

TRAIL_HW = {"path", "footway", "track", "bridleway", "steps"}
ROAD_HW  = {"primary", "secondary", "tertiary", "residential", "unclassified", "service"}
WATER_T  = {"water", "wetland"}
WATER_WW = {"river", "stream", "canal", "drain"}

if OSM_CACHE.exists():
    osm_data = pickle.load(open(OSM_CACHE, "rb"))
    n_t = len(osm_data["trails"])
    n_r = len(osm_data["roads"])
    n_w = len(osm_data["water"])
    print(f"OSM из кэша: trails={n_t:,} roads={n_r:,} water={n_w:,}")
else:
    # Шаг 1 - скачать PBF (~2 ГБ, один раз)
    if not PBF_PATH.exists():
        print("Скачиваем Alps PBF (~2 ГБ, 3-10 мин в зависимости от канала)...")
        print(f"URL: {PBF_URL}")
        import urllib.request
        seen_pct = [-1]
        def _progress(count, block, total):
            pct = int(count * block / total * 20) * 5
            if pct != seen_pct[0]:
                seen_pct[0] = pct
                print(f"  {pct}%  ({count*block/1e6:.0f} / {total/1e6:.0f} MB)")
        urllib.request.urlretrieve(PBF_URL, PBF_PATH, reporthook=_progress)
        print(f"Скачано: {PBF_PATH.stat().st_size/1e6:.0f} MB")
    else:
        print(f"PBF уже есть: {PBF_PATH.stat().st_size/1e6:.0f} MB")

    # Шаг 2 - парсим PBF через osmium (~2-3 мин)
    import osmium

    class AlpsWayHandler(osmium.SimpleHandler):
        def __init__(self):
            super().__init__()
            self.trails = []
            self.roads  = []
            self.water  = []

        def way(self, w):
            tags = w.tags
            hw  = tags.get("highway",  "")
            nat = tags.get("natural",  "")
            ww  = tags.get("waterway", "")
            try:
                coords = [(n.lon, n.lat) for n in w.nodes]
            except osmium.InvalidLocationError:
                return
            if len(coords) < 2:
                return
            if hw in TRAIL_HW:
                self.trails.append(coords)
            elif hw in ROAD_HW:
                self.roads.append(coords)
            elif nat in WATER_T or ww in WATER_WW:
                self.water.append(coords)

    print("Парсим PBF (2-3 мин)...")
    t0 = time.time()
    h = AlpsWayHandler()
    h.apply_file(str(PBF_PATH), locations=True)
    n_t = len(h.trails); n_r = len(h.roads); n_w = len(h.water)
    print(f"Готово за {time.time()-t0:.0f}с: trails={n_t:,} roads={n_r:,} water={n_w:,}")

    osm_data = {"trails": h.trails, "roads": h.roads, "water": h.water}
    pickle.dump(osm_data, open(OSM_CACHE, "wb"))
    print(f"Сохранено: {OSM_CACHE}")


## 3. STRtree — пространственный индекс

In [ ]:
import shapely as shp

def build_strtree(coords_list):
    # coords_list: list of [(lon, lat), ...]
    geoms = [LineString(c) for c in coords_list if len(c) >= 2]
    return STRtree(geoms), np.array(geoms)

print("Строим индексы...")
t0 = time.time()
trail_tree, trail_geoms = build_strtree(osm_data["trails"])
road_tree,  road_geoms  = build_strtree(osm_data["roads"])
water_tree, water_geoms = build_strtree(osm_data["water"])
print(f"Индексы: trails={len(trail_geoms):,} roads={len(road_geoms):,} water={len(water_geoms):,}  ({time.time()-t0:.1f}с)")

DEG2M_LAT = 111_320.0

def bulk_nearest_dist_m(tree, geoms_arr, lons, lats):
    pts = shp.points(lons, lats)
    idx = tree.nearest(pts)
    d_deg = shp.distance(pts, geoms_arr[idx])
    lat_m = d_deg * DEG2M_LAT
    lon_m = d_deg * DEG2M_LAT * np.cos(np.radians(lats))
    return np.sqrt(lat_m**2 + lon_m**2)

# Тест (Инсбрук)
print("Тест (Инсбрук 11.39E, 47.26N):")
def _test_one(tree, geoms_arr, lon, lat):
    idx = tree.nearest(shp.points([lon], [lat]))
    d = shp.distance(shp.points([lon], [lat]), geoms_arr[idx])[0]
    return d * DEG2M_LAT
print(f"  до тропы:  {_test_one(trail_tree, trail_geoms, 11.39, 47.26):.0f} м")
print(f"  до дороги: {_test_one(road_tree,  road_geoms,  11.39, 47.26):.0f} м")
print(f"  до воды:   {_test_one(water_tree, water_geoms, 11.39, 47.26):.0f} м")


## 4. Разметка сегментов

In [ ]:
LABELS_CACHE = CACHE_DIR / "hikr_osm_labels.pkl"

TRAIL_THRESH = 50
ROAD_THRESH  = 30
WATER_THRESH = 100

if LABELS_CACHE.exists():
    osm_labels = pickle.load(open(LABELS_CACHE, "rb"))
    print(f"Метки из кэша: {len(osm_labels['dist_trail_m']):,} сегментов")
else:
    lons = df["lon_mid"].values.astype(np.float64)
    lats = df["lat_mid"].values.astype(np.float64)
    n = len(df)
    print(f"Размечаем {n:,} сегментов (векторно)...")
    t0 = time.time()

    # Батч-обработка по 100k чтобы не исчерпать RAM
    BATCH = 100_000
    dist_trail = np.empty(n); dist_road = np.empty(n); dist_water = np.empty(n)
    for start in range(0, n, BATCH):
        end = min(start + BATCH, n)
        blon, blat = lons[start:end], lats[start:end]
        dist_trail[start:end] = bulk_nearest_dist_m(trail_tree, trail_geoms, blon, blat)
        dist_road[start:end]  = bulk_nearest_dist_m(road_tree,  road_geoms,  blon, blat)
        dist_water[start:end] = bulk_nearest_dist_m(water_tree, water_geoms, blon, blat)
        if (end // BATCH) % 5 == 0 or end == n:
            elapsed = time.time() - t0
            eta = elapsed / end * (n - end) if end < n else 0
            print(f"  {end:,}/{n:,}  {elapsed:.0f}с  ETA={eta:.0f}с")

    osm_labels = {
        "dist_trail_m": dist_trail,
        "dist_road_m":  dist_road,
        "dist_water_m": dist_water,
    }
    pickle.dump(osm_labels, open(LABELS_CACHE, "wb"))
    print(f"Готово за {(time.time()-t0)/60:.1f} мин")

df["dist_trail_m"] = osm_labels["dist_trail_m"]
df["dist_road_m"]  = osm_labels["dist_road_m"]
df["dist_water_m"] = osm_labels["dist_water_m"]

# Бинарные маски
df["on_trail"]   = (df["dist_trail_m"] < TRAIL_THRESH).astype(int)
df["on_road"]    = (df["dist_road_m"]  < ROAD_THRESH).astype(int)
df["near_water"] = (df["dist_water_m"] < WATER_THRESH).astype(int)
df["off_trail"]  = ((df["on_trail"] == 0) & (df["on_road"] == 0)).astype(int)

df["log_dist_trail"] = np.log1p(df["dist_trail_m"].clip(0, 5000))
df["log_dist_road"]  = np.log1p(df["dist_road_m"].clip(0, 5000))

print(f"on_trail:   {df['on_trail'].mean():.1%}")
print(f"on_road:    {df['on_road'].mean():.1%}")
print(f"off_trail:  {df['off_trail'].mean():.1%}")
print(f"near_water: {df['near_water'].mean():.1%}")
print(f"dist_trail_m: median={df['dist_trail_m'].median():.0f}м  p90={df['dist_trail_m'].quantile(.9):.0f}м")
print(f"dist_road_m:  median={df['dist_road_m'].median():.0f}м  p90={df['dist_road_m'].quantile(.9):.0f}м")

## 5. Кастомная сложность

In [ ]:
df["signed_slope_deg"] = np.degrees(np.arctan2(
    df["elev_diff_m"].values,
    np.maximum(df["dist_m"].values, 0.1)
))

slope = df["slope_deg"].values
road  = df["on_road"].values.astype(bool)
trail = df["on_trail"].values.astype(bool)
water = df["near_water"].values.astype(bool)

base = np.where(road,                 1,
       np.where(trail & (slope < 15), 2,
       np.where(trail | (slope < 25), 3,
                                      4)))
df["custom_difficulty"] = np.where(water & ~road, np.minimum(base+1, 5), base).astype(int)

print("custom_difficulty:")
print(df["custom_difficulty"].value_counts().sort_index())
print()
print("Сравнение с SAC (hikr):")
print(pd.crosstab(df["difficulty"], df["custom_difficulty"],
                  rownames=["SAC"], colnames=["custom"]))

In [ ]:
# -- Единый canonical split 70/15/15 (shared: NB-05, NB-06, NB-05b) --------
# Создаётся один раз, затем загружается всеми ноутбуками
_split_path = CACHE_DIR / 'canonical_split.pkl'
if _split_path.exists():
    _sp = pickle.load(open(_split_path, 'rb'))
    tr_ids, val_ids, te_ids = _sp['tr_ids'], _sp['val_ids'], _sp['te_ids']
    print(f'Loaded canonical split: train={len(tr_ids)} val={len(val_ids)} test={len(te_ids)} треков')
else:
    _uids = sorted(df['track_id'].unique().tolist())
    _rng  = np.random.default_rng(42)
    _idx  = np.arange(len(_uids))
    _rng.shuffle(_idx)
    _n_te  = max(1, int(len(_idx) * 0.15))
    _n_val = max(1, int(len(_idx) * 0.15))
    te_ids  = {_uids[i] for i in _idx[:_n_te]}
    val_ids = {_uids[i] for i in _idx[_n_te:_n_te+_n_val]}
    tr_ids  = {_uids[i] for i in _idx[_n_te+_n_val:]}
    pickle.dump({'tr_ids': tr_ids, 'val_ids': val_ids, 'te_ids': te_ids},
                open(_split_path, 'wb'))
    print(f'Created canonical split: train={len(tr_ids)} val={len(val_ids)} test={len(te_ids)} треков')

# Индексы сегментов
tr_mask  = df['track_id'].isin(tr_ids).values
val_mask = df['track_id'].isin(val_ids).values
te_mask  = df['track_id'].isin(te_ids).values
tr_idx   = np.where(tr_mask)[0]
val_idx  = np.where(val_mask)[0]
te_idx   = np.where(te_mask)[0]
print(f'Сегментов: train={len(tr_idx):,}  val={len(val_idx):,}  test={len(te_idx):,}')


## 6. Модель с кастомной сложностью

In [ ]:
FEATURES = [
    'signed_slope_deg', 'elev_diff_m',
    'on_trail', 'on_road', 'off_trail', 'near_water',
    'log_dist_trail', 'log_dist_road',
    'custom_difficulty',
]
CAT_F  = ['on_trail', 'on_road', 'off_trail', 'near_water', 'custom_difficulty']
TARGET = 'speed_kmh'

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df['track_id'].values

CB_PARAMS = dict(iterations=800, learning_rate=0.05, depth=6,
                 loss_function='MAE', eval_metric='MAE',
                 random_seed=42, verbose=0)

# -- 5-Fold CV на train (без val и test) ------------------------------------
gkf = GroupKFold(n_splits=5)
cv_mae, cv_r2 = [], []
print('GroupKFold CV (только на train-треках):')
for fold, (f_tr, f_val) in enumerate(gkf.split(X.iloc[tr_idx], y.iloc[tr_idx],
                                                 groups=groups[tr_idx])):
    m = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
    m.fit(X.iloc[tr_idx[f_tr]], y.iloc[tr_idx[f_tr]], cat_features=CAT_F,
          eval_set=(X.iloc[tr_idx[f_val]], y.iloc[tr_idx[f_val]]), use_best_model=True)
    yp = m.predict(X.iloc[tr_idx[f_val]])
    cv_mae.append(mean_absolute_error(y.iloc[tr_idx[f_val]], yp))
    cv_r2.append(r2_score(y.iloc[tr_idx[f_val]], yp))
    print(f'  Fold {fold+1}: MAE={cv_mae[-1]:.3f}  R^2={cv_r2[-1]:.3f}')
print(f'CV: MAE={np.mean(cv_mae):.3f}+/-{np.std(cv_mae):.3f}  R^2={np.mean(cv_r2):.3f}+/-{np.std(cv_r2):.3f}')

# -- Финальная base-модель: train -> val (early stop) -> test (eval only) -----
model = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
model.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CAT_F,
          eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
          use_best_model=True)

yp_tr = model.predict(X.iloc[tr_idx])
yp_te = model.predict(X.iloc[te_idx])
mae_te = mean_absolute_error(y.iloc[te_idx], yp_te)
r2_te  = r2_score(y.iloc[te_idx], yp_te)
print(f'\nBase model - Test: MAE={mae_te:.3f}  R^2={r2_te:.3f}')

model.save_model(str(MODELS_DIR / 'travel_time_custom.cbm'))
print('Saved: travel_time_custom.cbm')


## 8. Optuna — подбор гиперпараметров


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Canonical train/val - те же что и финальная модель (нет утечки test)
X_tr_o, y_tr_o   = X.iloc[tr_idx], y.iloc[tr_idx]
X_val_o, y_val_o = X.iloc[val_idx], y.iloc[val_idx]

def objective(trial):
    params = dict(
        iterations          = trial.suggest_int('iterations', 400, 1500),
        learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        depth               = trial.suggest_int('depth', 4, 8),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
        random_strength     = trial.suggest_float('random_strength', 0.0, 2.0),
        loss_function='MAE', eval_metric='MAE', random_seed=42, verbose=0,
        early_stopping_rounds=50,
    )
    m = CatBoostRegressor(**params)
    m.fit(X_tr_o, y_tr_o, cat_features=CAT_F,
          eval_set=(X_val_o, y_val_o), use_best_model=True)
    return mean_absolute_error(y_val_o, m.predict(X_val_o))

N_TRIALS = 40
print(f'Optuna: {N_TRIALS} trials...')
t0 = time.time()
study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Готово за {(time.time()-t0)/60:.1f} мин')

best = study.best_params
print(f'Best val MAE: {study.best_value:.4f}')
print('Best params:', best)

# Финальная opt-модель: train -> val (early stop) -> test (eval only)
model_opt = CatBoostRegressor(**best, loss_function='MAE', eval_metric='MAE',
                               random_seed=42, verbose=0, early_stopping_rounds=50)
model_opt.fit(X.iloc[tr_idx], y.iloc[tr_idx], cat_features=CAT_F,
              eval_set=(X.iloc[val_idx], y.iloc[val_idx]), use_best_model=True)

yp_te_opt = model_opt.predict(X.iloc[te_idx])
print(f'\nBase: MAE={mae_te:.3f}  R^2={r2_te:.3f}')
print(f'Opt:  MAE={mean_absolute_error(y.iloc[te_idx],yp_te_opt):.3f}  '
      f'R^2={r2_score(y.iloc[te_idx],yp_te_opt):.3f}')

model_opt.save_model(str(MODELS_DIR / 'travel_time_opt.cbm'))
print('Saved: travel_time_opt.cbm')


In [ ]:
def mape(y_true, y_pred):                   
      mask = y_true > 0.1                     
      return 100 * np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]))

print()
print(f"Base model: MAE={mae_te:.3f}  R^2={r2_te:.3f}  MAPE={mape_te:.1f}%")
print(f"Optuna opt: MAE={mean_absolute_error(y.iloc[te_idx],yp_te_opt):.3f}  "
      f"R^2={r2_score(y.iloc[te_idx],yp_te_opt):.3f}  MAPE={mape_te_opt:.1f}%")

    

## 7. Итог

In [ ]:
def mape(y_true, y_pred):
    mask = y_true > 0.1
    return 100 * np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask]))

mape_tr = mape(y.iloc[tr_idx].values, yp_tr)
mape_te = mape(y.iloc[te_idx].values, yp_te)

mae_tr = mean_absolute_error(y.iloc[tr_idx], yp_tr)
mae_te = mean_absolute_error(y.iloc[te_idx], yp_te)
r2_tr  = r2_score(y.iloc[tr_idx], yp_tr)
r2_te  = r2_score(y.iloc[te_idx], yp_te)

print("=" * 65)
print(f"Train: MAE={mae_tr:.3f}  R^2={r2_tr:.3f}  MAPE={mape_tr:.1f}%")
print(f"Test:  MAE={mae_te:.3f}  R^2={r2_te:.3f}  MAPE={mape_te:.1f}%")
print(f"Overfit: deltaMAE={mae_te-mae_tr:+.3f}  deltaMAPE={mape_te-mape_tr:+.1f}%")
print("=" * 65)
print()
cv_r2_mean  = np.mean(cv_r2)
cv_mae_mean = np.mean(cv_mae)
rows = [
    ("Baseline Tobler",        None,        1.186,       None),
    ("NB-03 SAC difficulty",   0.248,       1.065,       None),
    ("NB-05 custom difficulty", cv_r2_mean, cv_mae_mean, mape_te),
]
print(f"{'Модель':<35} {'CV R':>7}  {'CV MAE':>7}  {'MAPE':>7}")
print("-" * 60)
for name, r2, mae, mpe in rows:
    r2s  = f"{r2:.3f}"  if r2  is not None else "-"
    maes = f"{mae:.3f}" if mae is not None else "-"
    mpes = f"{mpe:.1f}%" if mpe is not None else "-"
    print(f"{name:<35} {r2s:>7}  {maes:>7}  {mpes:>7}")
print("=" * 60)
gap = 0.248 - cv_r2_mean
if abs(gap) < 0.015:
    print("Кастомная сложность ~ SAC (без user-меток)")
elif gap > 0:
    print(f"Потеря vs SAC: {gap:.3f} R^2")
else:
    print(f"Кастомная ЛУЧШЕ SAC на {-gap:.3f} R^2!")


In [ ]:
model_opt.save_model("models/travel_time_opt.cbm")